## B4 - Free / Blocked Classification
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 21st, 2026

### About: In this notebook you'll use the skills you just built in the last two notebooks to collect data and train a another AI. This time the AI will help pilot your robot!  
Try to look for the pattern in the work we're doing. We repeat the pattern in purpose so that you can develop a skill you'll retain and use.

### Bonus: you get to teleoperate your robot in this notebook!
Teleoperate - to remotely control a machine or robot, typically over a network.

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY B4.1: Learning to teleoperate your robot</span>


### In this simple activity you get time to practice controlling your robot with the xbox controller. Drive carefully!

Teleoperation relies on a sequence of hardware and software working together.
1. The xbox controller is connected to the robot's computer.
2. Special libraries for controllers and gamepads are used to interpret the signals from the controller.
3. Those signals are scaled and ajusted to suite the format for commanding the robot to move.
4. The scaled and adjusted signals are sent to the RVR.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####

# Step 1. Import Required Libraries
import traitlets
import ipywidgets.widgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Layout
from IPython.display import display
from ipyfilechooser import FileChooser
import Gamepad

from sphero_sdk import SpheroRvrObserver
from sphero_sdk import RawMotorModesEnum
from sphero_sdk import DriveFlagsBitmask
import time
import os

from jetcam_lite import TraitletCamera, bgr8_to_jpeg

# Import our shared robot connection helpers
from robot_utils import get_rvr, close_if_exists

# Import our shared gamepad + teleop control loop helpers
from gamepad_utils import connect_gamepad, start_control_loop, stop_control_loop

# Import our shared idempotency helpers for widgets/camera links
from jupyter_utils import register_click_handler, register_dlink

# Import our shared dataset helpers
from capture_utils import ensure_directory, save_image, count_images

# Import our shared training helpers
from train_utils import preview_dataset, prepare_dataloaders, build_model, train_model

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 2. Create the 'rvr' object using our shared robot_utils helper.

# get_rvr() connects to the robot and wakes it up. If you re-run this cell,
# or if a connection already exists, it safely reuses it instead of opening
# a second one.
rvr = get_rvr()

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 3. Check that the bluetooth Xbox controller is connected.
# Note: if the controller isn't found ensure your controller is on, otherwise you need to use the robot computer's graphical user interface to connect your controller.
!bluetoothctl devices

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 4. Connect the gamepad, and create sliders so you can see your stick inputs on screen.

# connect_gamepad() connects to the Xbox controller and returns two things:
#   gamepad       -- the connected gamepad object
#   gamepad_state -- an object whose .left / .right update live as you move the sticks
gamepad, gamepad_state = connect_gamepad()

# These sliders are for display only -- the control loop (next cells) is what
# actually reads gamepad_state and drives the robot.
left_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Left:',
                                   orientation='vertical', readout=True, readout_format='.2f')
right_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Right:',
                                    orientation='vertical', readout=True, readout_format='.2f')
display(widgets.HBox([left_slider, right_slider]))

### Why a single control loop?
### It might seem simpler to drive the motors directly every time a joystick moves, but that causes real problems: updating widgets from the gamepad's own background thread can freeze the display, and errors on a background thread can fail silently -- so a bad command might do nothing, with no warning at all.
### Instead, `start_control_loop()` below runs on its own dedicated thread, at a steady rate. It reads your current stick positions, updates the sliders above, and sends **one** combined drive command to the robot each cycle -- smoother, and safer to debug.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 5. Start driving! This starts the control loop -- drive with the gamepad now.

def update_sliders(left_value, right_value):
    left_slider.value = left_value
    right_slider.value = right_value

start_control_loop(rvr, gamepad_state, on_update=update_sliders)

### Go ahead and drive your robot around! Get a feel for how the sticks control it -- try turning in place, driving in a straight line, and stopping smoothly.
### When you're ready to move on, run the cell below to stop the control loop.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 6. Stop driving.
stop_control_loop()

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY B4.2: Free / Blocked Data Collection</span>

### Now you'll teleoperate your robot while collecting the data for your next AI model -- one that can tell whether the path ahead is **free** (clear to drive) or **blocked** (something in the way).
### Drive around freely. Whenever the path ahead looks clear, tap the **left bumper** to capture a `free` image. Whenever you're facing an obstacle, tap the **right bumper** to capture a `blocked` image -- think of the bumpers as a camera shutter button built into the controller.
### The on-screen buttons below do the exact same thing, in case driving and shooting with the controller at the same time is tricky at first -- use whichever is easier for you.
### 🎯 Goal: collect **at least 200 images of each class** before moving on.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 7. Set up the camera, dataset folders, and the data collection interface.

camera = TraitletCamera()
camera.start()

image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)
register_dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

# Creating a directory for our dataset and the individual directories for each class
data_dir = os.path.join(os.path.expanduser("~"), "Datasets", "free_blocked_dataset")
free_directory = os.path.join(data_dir, "free")
blocked_directory = os.path.join(data_dir, "blocked")

ensure_directory(free_directory)
ensure_directory(blocked_directory)

# Count boxes -- show how many images of each class you've collected so far
free_count = widgets.IntText(value=count_images(free_directory), description='free', layout=Layout(width='150px'))
blocked_count = widgets.IntText(value=count_images(blocked_directory), description='blocked', layout=Layout(width='150px'))

# On-screen fallback buttons, in case driving + gamepad-shutter together is tricky
free_button = widgets.Button(description='Capture Free', button_style='success')
blocked_button = widgets.Button(description='Capture Blocked', button_style='danger')

display(image_widget)
display(widgets.HBox([free_button, free_count]))
display(widgets.HBox([blocked_button, blocked_count]))

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 8. Define what happens on a capture request, and wire up both the
# gamepad bumpers and the on-screen buttons to trigger it.

def update_counts():
    free_count.value = count_images(free_directory)
    blocked_count.value = count_images(blocked_directory)

def capture_image(label):
    directory = free_directory if label == 'free' else blocked_directory
    save_image(image_widget.value, directory, label=label)
    update_counts()

def check_gamepad_captures(gamepad_state):
    """Checked every control loop cycle -- captures and clears any bumper
    request set since the last check."""
    if gamepad_state.capture_free_requested:
        gamepad_state.capture_free_requested = False
        capture_image('free')

    if gamepad_state.capture_blocked_requested:
        gamepad_state.capture_blocked_requested = False
        capture_image('blocked')

def on_free_button_clicked(button):
    capture_image('free')

def on_blocked_button_clicked(button):
    capture_image('blocked')

register_click_handler(free_button, on_free_button_clicked)
register_click_handler(blocked_button, on_blocked_button_clicked)

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 9. Start driving and collecting! Left bumper = free, right bumper = blocked.

def on_update(left_value, right_value):
    left_slider.value = left_value
    right_slider.value = right_value
    check_gamepad_captures(gamepad_state)

start_control_loop(rvr, gamepad_state, on_update=on_update)

### Drive around and collect at least 200 images of each class. Watch the counters above as you go.
### 🔬 Real Data Science: variety matters as much as quantity. Capture `blocked` images against different obstacles, angles, and distances, and `free` images from different parts of the room -- otherwise your model may just memorize your test course instead of learning the general idea of "path ahead" vs. "obstacle ahead."
### When you've collected enough, run the cell below to stop driving.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 10. Stop driving once you've collected enough images.
stop_control_loop()

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY B4.3: Train the Free / Blocked Classifier</span>

### Same pattern as B3a: look at your data, choose your hyperparameters, build your dataloaders and model, then train. See if you can get through this activity with less help than last time -- that's the whole point of the repetition!

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
preview_dataset(data_dir)

### Check your counts and sample images above -- at least 200 of each class, and a good variety of angles/lighting/obstacles. Collect more now if you need to; it's much cheaper than retraining later.

In [ ]:
##### ----- FEEL FREE TO CHANGE THESE VALUES ----- #####
model_name = "free_blocked_classifier_v1"
epochs = 15
learning_rate = 0.001
momentum = 0.9
batch_size = 64  # keep this at 8 or 16 -- see B3a notes on Jetson memory limits

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
train_loader, test_loader, class_names = prepare_dataloaders(
    data_dir,
    batch_size=batch_size,
    test_fraction=0.2
)

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
model, device = build_model(num_classes=len(class_names))

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
history, training_record = train_model(
    model,
    train_loader,
    test_loader,
    device,
    class_names,
    data_dir,
    model_name,
    epochs,
    learning_rate,
    momentum,
)

## Nice work! You've now collected, trained, and saved a Free/Blocked classifier -- the same model your robot will use next to start making its own driving decisions.

## **NEXT**: We'll evaluate this model and then start writing control logic that acts on what it predicts -- turning your robot from teleoperated to autonomous.